<a href="https://colab.research.google.com/github/junmk01/open-trading-api/blob/main/tinygpt_implementation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
# -*- coding: utf-8 -*-
"""
tiny_gpt.py — 나만의 TinyGPT (Notebook 06 기반)

전체 뼈대: Data -> Dataset -> Model -> Loss -> Train -> Sample
NB01~05는 이 구조를 이해하기 위한 단계별 설명서였고,
이 파일은 그 결론(NB06)을 정리/개선한 최종본이다.

[개선 포인트]
- 하이퍼파라미터를 맨 위에 모음 (실험하기 쉽게)
- train/val 분리로 과적합 확인 (eval loss 모니터링)
- 어떤 .txt든 학습 가능 (DATA_FILE만 바꾸면 됨)
- 생성 시 temperature / top-k 옵션 추가
"""

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from pathlib import Path


# =====================================================================
# 0. 하이퍼파라미터 (실험하기 쉽게 한곳에 모음)
# =====================================================================
DATA_FILE  = '/content/sample_data/For Whom the Bell Tolls.txt'   # 학습할 텍스트 파일. 다른 글로 바꾸려면 여기만 수정.

block_size = 64       # 한 샘플의 글자 수 (T) — 한 번에 보는 문맥 길이
batch_size = 64       # 한 번에 처리하는 샘플 수 (B)
emb_dim    = 128      # 글자 하나를 표현하는 숫자 개수 (C)
num_heads  = 4        # 어텐션 head 개수 (관점 수). emb_dim 이 num_heads 로 나눠떨어져야 함
num_layers = 4        # Block 을 몇 겹 쌓을지
dropout    = 0.1      # 과적합 방지 (학습 중 일부 연결을 랜덤하게 차단)
lr         = 3e-4     # 학습 속도 (보폭)
import os
max_epochs = int(os.environ.get("EPOCHS", 50))   # 학습 반복 횟수
max_steps  = int(os.environ.get("STEPS", 300))   # 한 epoch 당 step 수 (데이터가 커서 끊어줌)

device = "cuda" if torch.cuda.is_available() else "cpu"
torch.manual_seed(1337)   # 결과 재현용 (랜덤 고정)


# =====================================================================
# 1. 데이터 준비  (text -> stoi/itos -> data 텐서)
# =====================================================================
# 파일이 없을 때: 기본값(input.txt)이면 셰익스피어를 받아오고,
#               다른 파일명인데 없으면 안내 후 종료.
if not Path(DATA_FILE).exists():
    if DATA_FILE == "input.txt":
        import urllib.request
        print("input.txt 다운로드 중 (tiny shakespeare)...")
        url = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
        urllib.request.urlretrieve(url, "input.txt")
    else:
        raise FileNotFoundError(
            f"'{DATA_FILE}' 가 이 폴더에 없습니다. "
            f".txt 파일을 넣거나 DATA_FILE 값을 바꾸세요."
        )

# 인코딩 자동 처리 (utf-8 실패 시 latin-1 로 대체 — 어떤 txt 든 읽히게)
try:
    text = open(DATA_FILE, "r", encoding="utf-8").read()
except UnicodeDecodeError:
    text = open(DATA_FILE, "r", encoding="latin-1").read()

chars = sorted(list(set(text)))                 # 등장하는 모든 글자 (중복 제거)
stoi = {ch: i for i, ch in enumerate(chars)}    # 글자 -> 번호 (토크나이저)
itos = {i: ch for ch, i in stoi.items()}        # 번호 -> 글자
vocab_size = len(chars)

# 전체 텍스트를 번호의 긴 텐서로 (글자 -> 번호 단계. 임베딩은 모델 안에서.)
data = torch.tensor([stoi[ch] for ch in text], dtype=torch.long)

# train / val 분리 (앞 90% 학습, 뒤 10% 검증 — 과적합 확인용)
n = int(0.9 * len(data))
train_data = data[:n]
val_data   = data[n:]

print(f"파일: {DATA_FILE} | 텍스트 길이: {len(text):,} | vocab_size: {vocab_size}")
print(f"train: {len(train_data):,} | val: {len(val_data):,} | device: {device}")


# =====================================================================
# 2. Dataset  (NextTokenDataset: x, y 둘 다 시퀀스 — NB04/06)
# =====================================================================
class NextTokenDataset(Dataset):
    """긴 텐서에서 sliding window 로 (x, y) 를 떼어낸다.
    x = 연속된 block_size 글자
    y = x 를 한 칸 민 것 (각 위치의 '다음 글자')
    """
    def __init__(self, data, block_size):
        self.data = data
        self.block_size = block_size

    def __len__(self):
        return len(self.data) - self.block_size

    def __getitem__(self, idx):
        x = self.data[idx     : idx + self.block_size]
        y = self.data[idx + 1 : idx + self.block_size + 1]   # 한 칸 shift
        return x, y


train_ds = NextTokenDataset(train_data, block_size)
val_ds   = NextTokenDataset(val_data,   block_size)
train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
val_loader   = DataLoader(val_ds,   batch_size=batch_size, shuffle=True)


# =====================================================================
# 3. 모델 부품들 (NB06)
# =====================================================================
class Head(nn.Module):
    """single masked self-attention head.
    각 위치가 '앞 글자들'을 둘러보고 정보를 가중평균으로 끌어온다.
    """
    def __init__(self, emb_dim, head_size, block_size, dropout=0.1):
        super().__init__()
        self.key   = nn.Linear(emb_dim, head_size, bias=False)   # 내가 내거는 것
        self.query = nn.Linear(emb_dim, head_size, bias=False)   # 내가 찾는 것
        self.value = nn.Linear(emb_dim, head_size, bias=False)   # 내가 전달할 것
        # tril: 아래삼각행렬 = causal mask (미래 차단용). 학습 안 하는 고정값.
        self.register_buffer("tril", torch.tril(torch.ones(block_size, block_size)))
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        B, T, C = x.shape
        k = self.key(x)     # (B, T, head_size)
        q = self.query(x)
        v = self.value(x)

        # 1) 누구를 볼지 점수 = query · key  (방향이 맞을수록 큰 값)
        wei = q @ k.transpose(-2, -1) * (k.size(-1) ** -0.5)   # (B, T, T), √d 로 스케일
        # 2) 미래 가리기: 윗삼각(미래)을 -inf 로 -> softmax 후 0%
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float("-inf"))
        # 3) 점수를 비율로
        wei = F.softmax(wei, dim=-1)
        wei = self.dropout(wei)
        # 4) 그 비율대로 value 를 섞어 가져옴 (예측이 아니라 '재료 모으기')
        out = wei @ v       # (B, T, head_size)
        return out


class MultiHeadAttention(nn.Module):
    """Head 여러 개를 병렬로 -> 서로 다른 관점으로 동시에 문맥을 본다."""
    def __init__(self, emb_dim, num_heads, block_size, dropout=0.1):
        super().__init__()
        head_size = emb_dim // num_heads
        self.heads = nn.ModuleList([
            Head(emb_dim, head_size, block_size, dropout) for _ in range(num_heads)
        ])
        self.proj = nn.Linear(emb_dim, emb_dim)   # 이어붙인 head 들을 섞어 종합
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        out = torch.cat([h(x) for h in self.heads], dim=-1)   # head 결과 옆으로 붙임
        out = self.proj(out)
        out = self.dropout(out)
        return out


class FeedForward(nn.Module):
    """각 글자가 끌어모은 정보를 '혼자' 가공하는 층 (위치끼리 안 섞음)."""
    def __init__(self, emb_dim, dropout=0.1):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(emb_dim, 4 * emb_dim),   # 넓혔다가
            nn.ReLU(),                          # 비선형
            nn.Linear(4 * emb_dim, emb_dim),    # 다시 줄임
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)


class Block(nn.Module):
    """GPT 기본 빌딩 블록: (소통=attention) + (생각=feedforward), 둘 다 residual + LayerNorm."""
    def __init__(self, emb_dim, num_heads, block_size, dropout=0.1):
        super().__init__()
        self.ln1  = nn.LayerNorm(emb_dim)
        self.sa   = MultiHeadAttention(emb_dim, num_heads, block_size, dropout)
        self.ln2  = nn.LayerNorm(emb_dim)
        self.ffwd = FeedForward(emb_dim, dropout)

    def forward(self, x):
        # x + ... : residual (원본을 지름길로 더해 깊어도 안정적으로 학습)
        x = x + self.sa(self.ln1(x))     # 정규화 -> 소통 -> 원본 더하기
        x = x + self.ffwd(self.ln2(x))   # 정규화 -> 가공 -> 원본 더하기
        return x


class TinyGPT(nn.Module):
    def __init__(self, vocab_size, block_size, emb_dim, num_heads, num_layers, dropout):
        super().__init__()
        self.block_size = block_size
        self.token_embedding    = nn.Embedding(vocab_size, emb_dim)   # 글자 -> 벡터
        self.position_embedding = nn.Embedding(block_size, emb_dim)   # 위치 -> 벡터
        self.blocks = nn.Sequential(*[
            Block(emb_dim, num_heads, block_size, dropout) for _ in range(num_layers)
        ])
        self.ln_f    = nn.LayerNorm(emb_dim)
        self.lm_head = nn.Linear(emb_dim, vocab_size)   # 벡터 -> 다음 글자 점수

    def forward(self, x):
        B, T = x.shape
        pos = torch.arange(T, device=x.device)
        tok = self.token_embedding(x)            # (B, T, C)  내용
        pos = self.position_embedding(pos)[None] # (1, T, C)  위치
        h = tok + pos                            # 내용 + 위치
        h = self.blocks(h)                       # Block 여러 겹 통과
        h = self.ln_f(h)
        logits = self.lm_head(h)                 # (B, T, V)  각 위치의 다음 글자 점수
        return logits


# =====================================================================
# 4. Loss  (NB04~06)
# =====================================================================
def sequence_cross_entropy(logits, targets):
    # logits (B,T,V) -> (B,V,T) 로 바꿔야 cross_entropy 가 받음
    return F.cross_entropy(logits.transpose(1, 2), targets)


# =====================================================================
# 5. Train  (+ val 모니터링)
# =====================================================================
def train_one_epoch(model, loader, optimizer, device, max_steps=None):
    model.train()
    total_loss, total_count = 0.0, 0
    for step, (xb, yb) in enumerate(loader):
        xb, yb = xb.to(device), yb.to(device)
        logits = model(xb)                              # 1. 예측 (모든 위치 동시에)
        loss = sequence_cross_entropy(logits, yb)       # 2. 정답과 비교
        optimizer.zero_grad()                           # 3. 기울기 초기화
        loss.backward()                                 # 4. 역전파
        optimizer.step()                                # 5. 업데이트
        total_loss += loss.item() * xb.size(0)
        total_count += xb.size(0)
        if max_steps is not None and step + 1 >= max_steps:
            break
    return total_loss / total_count


@torch.no_grad()
def evaluate(model, loader, device, max_steps=50):
    model.eval()
    total_loss, total_count = 0.0, 0
    for step, (xb, yb) in enumerate(loader):
        xb, yb = xb.to(device), yb.to(device)
        loss = sequence_cross_entropy(model(xb), yb)
        total_loss += loss.item() * xb.size(0)
        total_count += xb.size(0)
        if step + 1 >= max_steps:
            break
    return total_loss / total_count


# =====================================================================
# 6. Sample  (autoregressive 생성: 한 글자씩 뽑아 뒤에 붙이며 반복)
# =====================================================================
@torch.no_grad()
def generate(model, start_text="\n", max_new_tokens=500, temperature=1.0, top_k=None):
    model.eval()
    # 시작 문구를 번호로 (없는 글자는 무시)
    idx = [stoi[c] for c in start_text if c in stoi]
    if len(idx) == 0:
        idx = [0]
    context = torch.tensor([idx], dtype=torch.long, device=device)

    for _ in range(max_new_tokens):
        # 입력이 block_size 보다 길면 뒤쪽만 사용 (position embedding 한계)
        context_cond = context[:, -model.block_size:]
        logits = model(context_cond)
        logits = logits[:, -1, :] / temperature     # 마지막 위치만, temperature 로 조절
        if top_k is not None:                       # top-k: 상위 k 개만 후보로
            v, _ = torch.topk(logits, top_k)
            logits[logits < v[:, [-1]]] = float("-inf")
        probs = F.softmax(logits, dim=-1)
        next_id = torch.multinomial(probs, num_samples=1)
        context = torch.cat([context, next_id], dim=1)

    return "".join(itos[i] for i in context[0].tolist())


# =====================================================================
# 실행
# =====================================================================
if __name__ == "__main__":
    model = TinyGPT(vocab_size, block_size, emb_dim, num_heads, num_layers, dropout).to(device)
    n_params = sum(p.numel() for p in model.parameters())
    print(f"모델 파라미터 수: {n_params:,}")

    optimizer = torch.optim.AdamW(model.parameters(), lr=lr)

    for epoch in range(max_epochs):
        train_loss = train_one_epoch(model, train_loader, optimizer, device, max_steps=max_steps)
        if epoch % 5 == 0 or epoch == max_epochs - 1:
            val_loss = evaluate(model, val_loader, device)
            print(f"epoch {epoch:3d} | train {train_loss:.4f} | val {val_loss:.4f}")

    print("\n===== 생성 결과 =====")
    sample = generate(model, start_text="\n", max_new_tokens=500, temperature=0.8, top_k=50)
    with open("sample_output.txt", "w", encoding="utf-8") as f:
        f.write(sample)
    print(sample)


파일: /content/sample_data/For Whom the Bell Tolls.txt | 텍스트 길이: 930,181 | vocab_size: 85
train: 837,162 | val: 93,019 | device: cuda
모델 파라미터 수: 821,845
epoch   0 | train 2.5490 | val 2.2479
epoch   5 | train 1.6121 | val 1.5220
epoch  10 | train 1.4445 | val 1.3777
epoch  15 | train 1.3646 | val 1.3168
epoch  20 | train 1.3182 | val 1.2908
epoch  25 | train 1.2826 | val 1.2584
epoch  30 | train 1.2607 | val 1.2415
epoch  35 | train 1.2386 | val 1.2409
epoch  40 | train 1.2207 | val 1.2266
epoch  45 | train 1.2066 | val 1.2180
epoch  49 | train 1.1958 | val 1.2220

===== 생성 결과 =====


AttributeError: 'OutStream' object has no attribute 'buffer'

In [4]:
print("\n===== 생성 결과 =====")
sample = generate(model, start_text="\n", max_new_tokens=5000, temperature=0.5, top_k=50)
with open("sample_output.txt", "w", encoding="utf-8") as f:
        f.write(sample)
print(sample)


===== 생성 결과 =====

	"There is no need and they were all right and all of the barrel and two and as the cave to the planes. But who was not a remain. It was always at the horse who was working and they were all right, but they came full on the bridge of the cave where the gun and the same of the bull past him and put his head and when he saw the later of his bowl face and the fascists with the cave started him for the time and he thought who were to the entrance that he was standing and one watched the same slope of the pillow stood and the brown on his head and he was standing and no put into the driver.
	"Then is the planes and strange here and went on the cave of the road and the bridge that flat of the same train. They have been gone to the tree and the lines was not to be and in his mouth of the road and the great and life and saw the bridge and took a man.
	"_Qué va_," said Robert Jordan. "It is a propect and the motorcycle of the small and the motorcycle of the difficulty when t

In [5]:
with open("sample_output.txt", "w", encoding="utf-8") as f:
        f.write(sample)